In [ ]:
import os
import json
import zipfile
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import Optional

# ==========================================
# 1. 데이터 구조 및 입력 추상화
# ==========================================

@dataclass
class NormalizedDocument:
    """
    모든 입력(일반 텍스트, 다양한 파일)이 최종적으로 변환되는 표준 텍스트 데이터 구조입니다.
    이후 도메인 추정기(DomainPredictor)는 파일 형식을 모르고 이 객체의 text 속성만 보고 판단합니다.
    """
    text: str
    source_type: str  # 'text', '.pdf', '.docx' 등
    file_path: Optional[str] = None

class InputSource:
    """
    입력된 데이터가 일반 텍스트인지, 파일 경로인지 자동으로 판별하고 추상화하는 클래스입니다.
    """
    def __init__(self, data: str):
        self.data = data
        self.is_file = self._check_if_file(data)

    def _check_if_file(self, text: str) -> bool:
        """
        입력된 문자열이 실제 존재하는 파일 경로인지 확인합니다.
        """
        # 1. 텍스트가 너무 길면(예: 2000자 이상) 파일 경로일 리가 없으므로 빠르게 텍스트로 판별
        if len(text) > 2000:
            return False
            
        # 2. 실제 디스크에 존재하는 파일인지 확인
        try:
            # os.path.isfile은 파일이 실제로 존재할 때만 True를 반환합니다.
            if os.path.isfile(text):
                return True
        except (ValueError, OSError):
            # 문자열에 경로로 사용할 수 없는 특수문자가 많거나 운영체제 제한을 넘으면
            # 에러가 발생할 수 있으므로 이를 무시하고 텍스트로 취급합니다.
            pass
            
        return False

# ==========================================
# 2. 포맷별 문서 추출기 (Extractors)
# ==========================================



class BaseExtractor:
    def extract(self, file_path: str) -> str:
        raise NotImplementedError

class TxtExtractor(BaseExtractor):
    def extract(self, file_path: str) -> str:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            return f.read()

class JsonExtractor(BaseExtractor):
    def extract(self, file_path: str) -> str:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            # 구조화된 JSON 데이터를 텍스트로 단순 변환 (필요시 특정 key만 파싱하도록 확장 가능)
            return json.dumps(data, ensure_ascii=False)

class PdfExtractor(BaseExtractor):
    def extract(self, file_path: str) -> str:
        # [Lazy Import] pypdf가 설치되지 않은 환경에서도 다른 포맷은 정상 동작하도록 내부 import
        import pypdf 
        
        text = ""
        with open(file_path, 'rb') as f:
            reader = pypdf.PdfReader(f)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
                    
        # TODO: 스캔된 PDF(이미지)인 경우 OCR(Tesseract 등) fallback 로직 추가 필요
        return text.strip()

class DocxExtractor(BaseExtractor):
    def extract(self, file_path: str) -> str:
        # [Lazy Import] python-docx 사용
        import docx
        
        doc = docx.Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])

class DocExtractor(BaseExtractor):
    def extract(self, file_path: str) -> str:
        """
        LibreOffice(soffice) 변환 기반으로 .doc를 임시 폴더에서 .docx로 변환 후 텍스트를 추출합니다.
        (시스템에 LibreOffice가 설치되어 있어야 합니다)
        """
        with tempfile.TemporaryDirectory() as temp_dir:
            try:
                subprocess.run([
                    'soffice', '--headless', '--convert-to', 'docx',
                    '--outdir', temp_dir, file_path
                ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                
                base_name = os.path.basename(file_path)
                docx_filename = os.path.splitext(base_name)[0] + ".docx"
                converted_path = os.path.join(temp_dir, docx_filename)
                
                if os.path.exists(converted_path):
                    # 생성된 docx를 DocxExtractor를 재사용하여 파싱
                    return DocxExtractor().extract(converted_path)
                else:
                    return ""
            except Exception as e:
                return f"[Error] .doc 변환 실패 (LibreOffice 확인 필요): {str(e)}"

class HwpxExtractor(BaseExtractor):
    
    
    def extract(self, file_path: str) -> str:
        """
        .hwpx는 내부적으로 zip 아카이브 및 xml 구조를 가지므로 이를 직접 파싱합니다.
        """
        text_content = []
        with zipfile.ZipFile(file_path, 'r') as zf:
            for filename in zf.namelist():
                # hwpx 텍스트 데이터는 보통 Contents/sectionN.xml 에 존재
                if filename.startswith("Contents/section") and filename.endswith(".xml"):
                    with zf.open(filename) as f:
                        tree = ET.parse(f)
                        root = tree.getroot()
                        # XML 태그에서 텍스트 노드 추출 (네임스페이스 관계없이 텍스트만 수집)
                        for elem in root.iter():
                            if elem.text and elem.text.strip():
                                text_content.append(elem.text.strip())
        return "\n".join(text_content)

# class HwpExtractor(BaseExtractor):
#     def extract(self, file_path: str) -> str:
#         # TODO: Windows + 한글 COM 자동화로 .hwpx 변환 후 재사용
#         # (요청에 따라 현재는 주석 처리 및 구현 생략)
#         raise NotImplementedError("hwp 파일 처리는 현재 미구현 상태입니다.")

# ==========================================
# 3. 문서 추출 계층 (Universal Extractor)
# ==========================================

class UniversalDocumentExtractor:
    """
    InputSource를 받아 파일 확장자에 맞는 Extractor를 매핑하여
    최종적으로 NormalizedDocument 객체로 반환하는 클래스입니다.
    """
    def __init__(self):
        self.extractors = {
            '.txt': TxtExtractor(),
            '.json': JsonExtractor(),
            '.pdf': PdfExtractor(),
            '.docx': DocxExtractor(),
            '.doc': DocExtractor(),
            '.hwpx': HwpxExtractor(),
            # '.hwp': HwpExtractor(),
        }

    def process(self, source: InputSource) -> NormalizedDocument:
        # 1. 파일이 아닌 직접 입력된 텍스트인 경우
        if not source.is_file:
            return NormalizedDocument(text=source.data, source_type="text")
        
        # 2. 파일인 경우 포맷별 추출
        _, ext = os.path.splitext(source.data)
        ext = ext.lower()
        
        if ext in self.extractors:
            extracted_text = self.extractors[ext].extract(source.data)
            return NormalizedDocument(text=extracted_text, source_type=ext, file_path=source.data)
        else:
            raise ValueError(f"지원하지 않는 파일 형식입니다: {ext}")

# ==========================================
# 실행 테스트 코드 (모듈 테스트용)
# ==========================================
if __name__ == "__main__":
    # 임포트 오류 방지를 위해 위에 작성했던 UniversalDocumentExtractor 코드가 함께 있어야 합니다.
    extractor = UniversalDocumentExtractor()
    
    # 테스트 1: 직접 텍스트 입력 (is_file 생략)
    text_data = "A기관이 10억을 투자하기로 했다."
    text_input = InputSource(text_data)
    doc_text = extractor.process(text_input)
    print(f"[테스트 1] 타입: {doc_text.source_type}")
    print(f"추출 결과: {doc_text.text}\n")
    
    # 테스트 2: 파일 경로 입력 (is_file 생략, 실제 경로로 테스트)
    # 실제 존재하는 파일 경로를 넣으면 자동으로 파일로 인식합니다.
    file_path = r"C:\Users\PLATiNA01\OneDrive - 한림대학교\바탕 화면\캡스톤 코드\2월1주차.pdf"
    
    # 파일이 존재하는 경우에만 테스트를 실행하도록 방어 코드 작성
    if os.path.exists(file_path):
        file_input = InputSource(file_path)
        doc_file = extractor.process(file_input)
        print(f"[테스트 2] 타입: {doc_file.source_type}")
        print(f"추출 결과 길이: {doc_file.text}")
    else:
        print(f"[테스트 2] '{file_path}' 파일이 존재하지 않아 파일 테스트는 건너뜁니다.")

[테스트 1] 타입: text
추출 결과: A기관이 10억을 투자하기로 했다.

[테스트 2] 타입: .pdf
추출 결과 길이: 핵심파트너
병원, 
카드사, 
공공기관
핵심활동
무결성 검증,
가치제안
파일 무결성 검증을 
통한 자료 정당성 증
명
고객관계
암호학적 
리포트 제공
고객
학교,
공공기관,
보험사,
기업
핵심자원
진단서, 영수증,
거래내역서 등
증빙 자료,
무결성 검증 
소프트웨어
채널
앱 / 웹
비용
파트너 유지 비용,
온라인 서비스 제공 시 서버 유지 비용
수익
구독 서비스
시장 조사
비즈플레이
-무증빙 서비스 제공
-회사마다의 경비사용 규정을 환경설정APP을 이용하여 제공함으로써, 비용 규정 효과적으로 관리
경비사용 규정 설정
피드백 반영(문제점)
1. 비즈니스화
– 캡스톤 주제가 아닌 연구 목적이 강함
-제안: 고객에게 구독 서비스를 제공
자료 정당성 여부 증명 건당 비용 청구
(건당 얼마를 받을 것이며, 이게 마진율이 진행 가치가 있는가?)
2. 현장 상황(Trend) 
– 파일 주고 받는 것으로 증빙하지 않음
ex)크로아티아 – ‘실시간 영수증 검증’ 시스템 ‘피스칼리자치야’ Fiskalizacija
목적: 탈세 방지 - 크로아티아의 현금 거래 비중이 매우 높다는 경제적 특성으로 인한 잦은 탈세(vs 카드 사용이 잦은 한국)
-모든 POS 거래에 대해 세무 당국 서버에서 실시간 인증 및 발급되는 영수증 시스템 법제화
-파일 전송 대신 증명값, 서버 인증으로 진위 확인 구조
ex)한국 - 공공 마이데이터 서비스 API
-문서 파일을 직접 받는 것이 아니라 API 형태로 근거/출처를 조회/검증하는 방식
-새로운 위변조 기술은 결국 등장함
-제안: 우리가 제공하는 서비스는 단순하게 위변조를 방지하는 것이 아닌, 악의적인 유저가 자료를 위변조한 경우, 무결성 검증을 통해
해당 자료가 유효하지 않다는 정보를 제공

3. 현실성
– 현실성에 대한 설명 부족, 상용화를 위해 필요한 것들을 강제화할 수 있는가?
-우려사항:
1) 신뢰기관에서 텍스트 워터

In [19]:
from dataclasses import dataclass
from typing import List, Dict, Any

# ==========================================
# 1. 출력 결과 데이터 구조
# ==========================================
@dataclass
class DomainExtractionResult:
    """
    최종 도메인 추정 결과를 담는 데이터 구조입니다.
    이후 마스킹 엔진이나 외부 LLM으로 전달될 때 활용됩니다.
    """
    predicted_domain: str                  # 가장 확률이 높은 최종 도메인
    confidence: float                      # 해당 도메인의 확신도 (0.0 ~ 1.0)
    top_k_candidates: List[Dict[str, Any]] # 상위 후보 도메인들과 점수 리스트
    reasoning: str                         # 판단 근거 설명

# ==========================================
# 2. 도메인 예측기 인터페이스 및 구현
# ==========================================
class BaseDomainPredictor:
    def predict(self, text: str) -> DomainExtractionResult:
        raise NotImplementedError

class ZeroShotDomainPredictor(BaseDomainPredictor):
    """
    Hugging Face의 Zero-Shot Classification 파이프라인을 활용하여
    텍스트를 사전에 정의된 추상적인 도메인 카테고리로 분류합니다.
    """
    def __init__(self, candidate_labels: List[str] = None):
        # 외부에서 라벨을 주입하지 않으면 기본 회의 도메인 템플릿 사용
        self.candidate_labels = candidate_labels or [
            "투자 및 재무", 
            "인사 및 채용", 
            "법률 및 계약", 
            "연구 및 기술", 
            "식사 및 일상 대화"
        ]
        self.classifier = None  # 모델 지연 로딩(Lazy Loading)을 위해 None으로 초기화

    def _load_model(self):
        """
        실제 예측이 실행될 때 한 번만 무거운 모델을 메모리에 올립니다.
        """
        if self.classifier is None:
            try:
                # [Lazy Import] 무거운 transformers 라이브러리를 내부에서 import
                from transformers import pipeline
                
                print("[System] Zero-Shot 분류 모델을 로드하는 중입니다. (최초 1회만 실행됨, 약간의 시간이 소요될 수 있습니다...)")
                
                # 다국어(한국어 포함) 성능이 우수한 자연어 추론(NLI) 모델 사용
                self.classifier = pipeline(
                    "zero-shot-classification", 
                    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
                )
                print("[System] 모델 로드 완료!")
                
            except ImportError:
                raise ImportError(
                    "[Error] 'transformers'와 'torch' 라이브러리가 필요합니다. \n"
                    "터미널에서 'pip install transformers torch'를 실행해주세요."
                )

    def predict(self, text: str) -> DomainExtractionResult:
        # 모델 로드 (이미 로드되어 있으면 패스)
        self._load_model()
        
        # 1. 텍스트 길이 제한 (토큰 한도 초과 방지)
        # 긴 회의록의 경우, 앞부분 1500자 정도만 읽어도 전체 도메인을 추정하는 데 충분합니다.
        max_length = 1500
        truncated_text = text[:max_length]
        
        # 내용이 비어있을 경우 방어 코드
        if not truncated_text.strip():
            return DomainExtractionResult("알 수 없음", 0.0, [], "텍스트가 비어있어 판단할 수 없습니다.")
            
        # 2. 제로샷 분류 실행
        result = self.classifier(truncated_text, self.candidate_labels)
        
        # 3. 결과 파싱
        labels = result['labels']
        scores = result['scores']
        
        predicted_domain = labels[0]
        confidence = scores[0]
        
        top_k_candidates = [
            {"domain": label, "score": round(score, 4)} 
            for label, score in zip(labels, scores)
        ]
        
        # 4. 판단 근거 생성
        reasoning = f"주어진 {len(self.candidate_labels)}개의 도메인 카테고리 중, 텍스트 맥락상 '{predicted_domain}'에 해당할 확률이 {confidence*100:.1f}%로 가장 높게 평가되었습니다."
        
        return DomainExtractionResult(
            predicted_domain=predicted_domain,
            confidence=confidence,
            top_k_candidates=top_k_candidates,
            reasoning=reasoning
        )

# ==========================================
# 3. 실행 테스트 코드
# ==========================================
if __name__ == "__main__":
    # 1. 예측기 인스턴스 생성 (이 시점에서는 모델을 로드하지 않음)
    predictor = ZeroShotDomainPredictor(
        candidate_labels=["투자 유치", "인사 고과", "점심 메뉴", "기술 개발"]
    )
    
    # 2. 이전 단계에서 추출했다고 가정하는 텍스트(NormalizedDocument.text)
    sample_extracted_text = "오늘 춘천에서 점심 먹으려는데 장소 정하는 회의할거야."
    
    print("--- 예측 시작 ---")
    print("입력텍스트: ", sample_extracted_text)
    
    # 3. 도메인 예측 (이때 모델 로드됨)
    result = predictor.predict(sample_extracted_text)
    
    print("\n[최종 예측 결과]")
    print(f"도메인: {result.predicted_domain}")
    print(f"확신도: {result.confidence:.4f}")
    print(f"근거: {result.reasoning}")
    print("\n[상위 후보군]")
    for cand in result.top_k_candidates:
        print(f" - {cand['domain']}: {cand['score']:.4f}")

--- 예측 시작 ---
입력텍스트:  오늘 춘천에서 점심 먹으려는데 장소 정하는 회의할거야.
[System] Zero-Shot 분류 모델을 로드하는 중입니다. (최초 1회만 실행됨, 약간의 시간이 소요될 수 있습니다...)


Device set to use cuda:0


[System] 모델 로드 완료!

[최종 예측 결과]
도메인: 점심 메뉴
확신도: 0.9867
근거: 주어진 4개의 도메인 카테고리 중, 텍스트 맥락상 '점심 메뉴'에 해당할 확률이 98.7%로 가장 높게 평가되었습니다.

[상위 후보군]
 - 점심 메뉴: 0.9867
 - 인사 고과: 0.0095
 - 투자 유치: 0.0031
 - 기술 개발: 0.0008
